# Fine-tune Cross-Encoder v0.6 - 20 Epochs with Early Stopping

Phase 4 of experimentation roadmap: Test 20 epochs with early stopping to prevent overfitting.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **20** (extended from 15) |
| **Batch size** | 16 (locked from Phase 2.1) |
| **Learning rate** | 5e-5 (optimal from Phase 2.1) |
| **Early Stopping** | patience=3 (stop if val LabelAcc doesn't improve for 3 epochs) |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Test if extended training improves test LabelAcc over current baseline (65.15%).

**Expected**: +1-2pp improvement → 66-67% via early stopping that saves best epoch.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions

In [ ]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

## Load Dataset

In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Training with Early Stopping

In [ ]:
import torch
from torch.utils.data import DataLoader
import os

# Configuration
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
learning_rate = 5e-5
epochs = 20
batch_size = 16
patience = 3  # Early stopping patience

run_name = "v0.6-mse-spearman-20ep-early-stopping"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

# Calculate warmup steps
total_steps = (len(train_examples) // batch_size + 1) * epochs
warmup_steps = int(total_steps * 0.1)

print(f"📊 Training Configuration:")
print(f"  Base model: {base_model}")
print(f"  Learning rate: {learning_rate:.0e}")
print(f"  Epochs: {epochs} (with early stopping, patience={patience})")
print(f"  Batch size: {batch_size}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total steps: {total_steps}")
print()

# Initialize model
model = CrossEncoder(
    base_model,
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Setup evaluator and data
base_evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# Wrapper for early stopping
class EarlyStoppingEvaluator:
    def __init__(self, base_eval, patience=3):
        self.base_eval = base_eval
        self.patience = patience
        self.best_score = -float('inf')
        self.patience_counter = 0
        self.should_stop = False

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        score = self.base_eval(model, output_path, epoch, steps)
        
        if score > self.best_score:
            self.best_score = score
            self.patience_counter = 0
            print(f"  ✅ Epoch {epoch}: score improved to {score:.4f}")
        else:
            self.patience_counter += 1
            print(f"  ⚠️  Epoch {epoch}: no improve (patience {self.patience_counter}/{self.patience})")
            if self.patience_counter >= self.patience:
                self.should_stop = True

        return score

evaluator = EarlyStoppingEvaluator(base_evaluator, patience=patience)

print(f"🚀 Training...")

# Train (early stopping checked via evaluator)
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=epochs,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': learning_rate},
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True,
    evaluation_steps=int(len(train_dataloader) * 0.1) + 1
)

if evaluator.should_stop:
    print(f"\n✅ Training stopped early at best epoch!")
else:
    print(f"\n✅ Training completed all {epochs} epochs!")

## Evaluation

In [ ]:
# Load best model and compute metrics
best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n📊 Results:")
print(f"  Val LabelAcc:  {val_metrics['LabelAcc']:.4f} ({val_metrics['LabelAcc']*100:.2f}%)")
print(f"  Val MAE:       {val_metrics['MAE']:.4f}")
print(f"  Val RMSE:      {val_metrics['RMSE']:.4f}")
print()
print(f"  Test LabelAcc: {test_metrics['LabelAcc']:.4f} ({test_metrics['LabelAcc']*100:.2f}%)")
print(f"  Test MAE:      {test_metrics['MAE']:.4f}")
print(f"  Test RMSE:     {test_metrics['RMSE']:.4f}")
print()
print(f"💡 Baseline (15 epochs): 65.15%")
improvement = (test_metrics['LabelAcc'] - 0.6515) * 100
print(f"   Change: {improvement:+.2f}pp")

## Save Report

In [ ]:
import os
import json

os.makedirs('artifacts/reports', exist_ok=True)

report = {
    "experiment": "Phase 4: 20 Epochs with Early Stopping",
    "base_model": base_model,
    "dataset_version": "v0.5",
    "dataset_size": {
        "train": len(train_examples),
        "val": len(val_examples),
        "test": len(test_examples),
        "total": len(train_examples) + len(val_examples) + len(test_examples)
    },
    "run": run_name,
    "loss": "MSE",
    "evaluator": "Spearman",
    "epochs": epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "early_stopping": {
        "enabled": True,
        "patience": patience
    },
    "metrics": {
        "validation": {
            "MAE": float(val_metrics['MAE']),
            "RMSE": float(val_metrics['RMSE']),
            "LabelAcc": float(val_metrics['LabelAcc'])
        },
        "test": {
            "MAE": float(test_metrics['MAE']),
            "RMSE": float(test_metrics['RMSE']),
            "LabelAcc": float(test_metrics['LabelAcc'])
        }
    },
    "model_path": f"artifacts/models/{output_dir.split('/')[-1]}",
    "comparison_to_baseline": {
        "baseline_test_labelacc": 0.6515,
        "improvement_pp": float(improvement)
    }
}

report_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_20ep_early_stopping_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✅ Report saved: {report_path}")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save model
src_dir = output_dir
dest_dir = f"{drive_base}/models/{output_dir.split('/')[-1]}"
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
shutil.copytree(src_dir, dest_dir)
print(f"✅ Saved model: {output_dir.split('/')[-1]}")

# Save report
dest_report = f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_20ep_early_stopping_report.json"
shutil.copy(report_path, dest_report)
print(f"✅ Saved report: fine_tune_cross_encoder_v0.6_20ep_early_stopping_report.json")

print(f"\n✅ All saved to Google Drive!")